# ÉquiAlgo · labo d'hypothèses (Colab, CPU, < 1 min)

**Pas besoin de GPU.** 10 000 lignes × 9 colonnes : tout tourne en ~30 s sur CPU.
`Exécution > Modifier le type d'exécution > CPU`, puis `Exécution > Tout exécuter`.
Votre soumission actuelle : téléversez-la (panneau Fichiers) sous le nom `predictions_actuelles.csv`.

Ce que fait ce carnet :
1. Ajuste la règle du comité (régression logistique, < 1 s) et isole la pénalité régionale.
2. Écrit des **sondes** (`sondes/*.csv`) : chacune est une allocation candidate (une hypothèse sur l'étalon).
3. Vous soumettez une sonde sur HxBuddy, vous recopiez **accuracy et F1** dans `HXBUDDY` (section « Saisie HxBuddy »).
4. Le carnet décode le **nombre de positifs de l'étalon** à partir de (accuracy, F1), ajuste une famille
   d'étalons `coteR + a·heures + b·log(revenu) + c·premiere_gen + bruit`, taux q, et propose la sonde suivante.
5. Il écrit `predictions.csv` (meilleure allocation, budget vérifié) et trace le front de Pareto.

Pourquoi HxBuddy mesure l'étalon et pas le comité : le meilleur modèle possible contre `decision_octroi`
plafonne à ~89 % (bruit du comité). Un score de 94,7 % ne peut venir que de l'étalon caché.

In [ ]:
import glob, json, os, sys, subprocess, time
T0 = time.time()
IN_COLAB = 'google.colab' in sys.modules

DATA_DIR = os.environ.get('EQUIALGO_DATA')
for cand in ([] if DATA_DIR else ['data', 'equialgo-participants/data', 'codeml2026/equialgo-participants/data']):
    if os.path.exists(os.path.join(cand, 'donnees_demandes.csv')):
        DATA_DIR = cand
        break
if DATA_DIR is None and IN_COLAB:
    # Dépôt public : clone direct. Dépôt privé : le clone échoue et on passe au téléversement.
    subprocess.run(['git', 'clone', '-q', '--depth', '1',
                    'https://github.com/Francklin9999/codeml2026'], check=False)
    if os.path.exists('codeml2026/equialgo-participants/data/donnees_demandes.csv'):
        DATA_DIR = 'codeml2026/equialgo-participants/data'
if DATA_DIR is None and IN_COLAB:
    from google.colab import files
    print('Téléversez donnees_demandes.csv et candidats_evaluation.csv')
    files.upload()
    DATA_DIR = '.'
print('Données :', DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

demandes = pd.read_csv(f'{DATA_DIR}/donnees_demandes.csv')
candidats = pd.read_csv(f'{DATA_DIR}/candidats_evaluation.csv')
ELOIGNEES = ['Bas-Saint-Laurent', 'Cote-Nord', 'Gaspesie-Iles-de-la-Madeleine']
N = len(candidats)
BUDGET_MIN, BUDGET_MAX = 0.36, 0.44
K_MIN, K_MAX = int(np.ceil(BUDGET_MIN * N)) + 10, int(np.floor(BUDGET_MAX * N)) - 10  # marge de sécurité
ECART_BASE = 0.270  # écart d'égalité des chances du modèle de base, donné par le règlement


def variables_comite(df):
    return pd.DataFrame({
        'cote_r': df.cote_r_equivalent,
        'heures': df.heures_travail_semaine,
        'log_revenu': np.log(df.revenu_familial_estime),
        'eloignee': df.region_administrative.isin(ELOIGNEES).astype(int),
    })


# Règle du comité : un score linéaire commun + une pénalité constante pour les régions éloignées.
# (Programme, distance, première génération : coefficients ~0 une fois la région présente.)
comite = LogisticRegression(C=1e4, max_iter=10000).fit(variables_comite(demandes), demandes.decision_octroi)
W = dict(zip(variables_comite(demandes).columns, comite.coef_[0]))
A_COMITE = W['heures'] / W['cote_r']       # points de cote R par heure travaillée
B_COMITE = W['log_revenu'] / W['cote_r']   # points de cote R par unité de log(revenu)
PENALITE = W['eloignee'] / W['cote_r']     # points de cote R retirés aux régions éloignées
print('Logit du comité :', {k: round(v, 3) for k, v in W.items()})
print(f'En points de cote R : heures {A_COMITE:+.3f}/h, log(revenu) {B_COMITE:+.3f}, '
      f'pénalité éloignée {PENALITE:+.2f} points')

g = candidats.region_administrative.isin(ELOIGNEES).astype(int).to_numpy()  # 1 = éloignée
r = candidats.cote_r_equivalent.to_numpy()
h = candidats.heures_travail_semaine.to_numpy().astype(float)
li = np.log(candidats.revenu_familial_estime.to_numpy())
fg = candidats.premiere_generation_universitaire.to_numpy().astype(float)

In [ ]:
# Modèle de base du règlement (forêt de la cellule finale de baseline_model.ipynb).
# C'est par rapport à lui que l'écart 0,270 est mesuré : il sert d'ancre.
CAT = ['programme_etudes', 'region_administrative', 'code_postal_3']
X_hist = pd.get_dummies(demandes.drop(columns=['id_candidat', 'decision_octroi']), columns=CAT)
X_cand = pd.get_dummies(candidats.drop(columns=['id_candidat']), columns=CAT).reindex(columns=X_hist.columns, fill_value=0)
base = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42, n_jobs=-1) \
    .fit(X_hist, demandes.decision_octroi).predict(X_cand).astype(int)
print(f'Base : octroi {base.mean():.1%} | centres {base[g == 0].mean():.1%} | éloignées {base[g == 1].mean():.1%}')

In [ ]:
def score(a, b, c=0.0):
    """Score de mérite en points de cote R."""
    return r + a * h + b * li + c * fg


def top_k(s, k):
    pred = np.zeros(N, dtype=int)
    pred[np.argsort(-s, kind='stable')[:k]] = 1
    return pred


def ecrire(pred, chemin):
    pd.DataFrame({'id_candidat': candidats.id_candidat, 'decision_octroi': pred}).to_csv(chemin, index=False)


def lire(chemin):
    s = pd.read_csv(chemin).set_index('id_candidat').loc[candidats.id_candidat, 'decision_octroi']
    return s.to_numpy().astype(int)


SONDES_FIXES = {
    'P0_contrefactuel_40': top_k(score(A_COMITE, B_COMITE), 1600),       # comité sans pénalité (strat1)
    'P1_contrefactuel_44': top_k(score(A_COMITE, B_COMITE), 1750),       # même rang, taux max
    'P2_coteR_seule_40': top_k(r, 1600),
    'P3_coteR_heures_40': top_k(score(A_COMITE, 0), 1600),               # sans revenu
    'P4_coteR_revenu_40': top_k(score(0, B_COMITE), 1600),               # sans heures
    'P5_premiere_gen_40': top_k(score(A_COMITE, B_COMITE, 1.0), 1600),   # bonus 1re génération
}
os.makedirs('sondes', exist_ok=True)
for nom, pred in SONDES_FIXES.items():
    if not os.path.exists(f'sondes/{nom}.csv'):
        ecrire(pred, f'sondes/{nom}.csv')

# Tout fichier déposé dans sondes/ est modélisé (y compris les « suivante_* » écrits plus bas).
SONDES = {os.path.basename(f)[:-4]: lire(f) for f in sorted(glob.glob('sondes/*.csv'))}

# Votre soumission actuelle (celle à 94,7 %) : déposez-la à côté du carnet sous ce nom.
MON_FICHIER = 'predictions_actuelles.csv'
if os.path.exists(MON_FICHIER):
    SONDES['MIEN'] = lire(MON_FICHIER)
if IN_COLAB:   # panneau Fichiers > sondes.zip > Télécharger
    import shutil
    shutil.make_archive('sondes', 'zip', 'sondes')
for nom, pred in SONDES.items():
    print(f'{nom:28s} octroi {pred.mean():.1%} | centres {pred[g == 0].mean():.1%} | éloignées {pred[g == 1].mean():.1%}')

In [ ]:
# Moteur : pour chaque hypothèse (a, b, c, sigma), un seul tri ; tous les taux q se lisent par sommes cumulées.
Q_GRID = np.round(np.arange(0.34, 0.4801, 0.005), 3)
K_GRID = np.round(Q_GRID * N).astype(int)
# Les poids exacts du comité sont ajoutés à la grille : c'est l'hypothèse de tête.
A_GRID = np.unique(np.round(np.r_[np.arange(0.0, 0.401, 0.025), A_COMITE], 3))
B_GRID = np.unique(np.round(np.r_[np.arange(-1.0, 2.51, 0.125), B_COMITE], 3))
C_GRID = [0.0, 0.5, 1.0]
SIGMA_GRID = [0.0, 0.5, 1.0]   # bruit de l'étalon, en points de cote R
N_TIRAGES = 6
BRUIT = np.random.default_rng(0).standard_normal((N_TIRAGES, N))


def evaluer(order, P):
    """acc[sonde, q] et écart EO de la base pour un ordre de mérite donné (étalon = top-k de cet ordre)."""
    tp = np.cumsum(P[:, order], axis=1)[:, K_GRID - 1]
    acc = (2 * tp + N - P.sum(1, keepdims=True) - K_GRID) / N
    gs, bs = g[order], base[order]
    tpr_c = np.cumsum(bs * (1 - gs))[K_GRID - 1] / np.cumsum(1 - gs)[K_GRID - 1]
    tpr_r = np.cumsum(bs * gs)[K_GRID - 1] / np.cumsum(gs)[K_GRID - 1]
    return acc, tpr_c - tpr_r


def table_hypotheses(sondes):
    noms = list(sondes)
    P = np.array([sondes[n] for n in noms])
    lignes = []
    for a in A_GRID:
        for b in B_GRID:
            for c in C_GRID:
                s0 = score(a, b, c)
                for sig in SIGMA_GRID:
                    tirages = [s0] if sig == 0 else [s0 + sig * e for e in BRUIT]
                    res = [evaluer(np.argsort(-s, kind='stable'), P) for s in tirages]
                    acc = np.mean([x[0] for x in res], axis=0)
                    gap = np.mean([x[1] for x in res], axis=0)
                    for j, q in enumerate(Q_GRID):
                        lignes.append([a, b, c, sig, q, gap[j], *acc[:, j]])
    return pd.DataFrame(lignes, columns=['a', 'b', 'c', 'sigma', 'q', 'ecart_base', *noms])


t = time.time()
HYP = table_hypotheses(SONDES)
print(f'{len(HYP):,} hypothèses évaluées en {time.time() - t:.1f} s')

## Saisie HxBuddy
Une ligne par fichier soumis : `'nom': (accuracy, F1 macro)`, tels qu'affichés (91.93, 91.7) ou en fraction.
Mettez `None` pour le F1 si vous ne l'avez pas noté. Ensuite : `Exécution > Exécuter les cellules suivantes`
depuis la cellule ci-dessous (~10 s, le calcul des hypothèses est réutilisé).

**Mode simulation** : remplissez `SIMULATION` pour vérifier, sans dépenser de soumission, que la procédure
retrouve un étalon connu. Les scores simulés sont arrondis comme HxBuddy (3 décimales).

In [ ]:
HXBUDDY = {
    'P1_contrefactuel_44': (91.93, 91.70),
    'suivante_a0.0_b-1.0_c0.5_k1600': (90.83, 90.44),
    # 'MIEN': (94.7, None),
    # 'P3_coteR_heures_40': (0.0, 0.0),
}

SIMULATION = json.loads(os.environ.get('EQUIALGO_SIMULATION', 'null'))
# SIMULATION = dict(a=0.145, b=1.287, c=0.0, sigma=0.0, q=0.45,
#                   sondes=['P0_contrefactuel_40', 'P1_contrefactuel_44', 'P3_coteR_heures_40'])

ETALON_SIMULE = None
if SIMULATION:
    s_sim = score(SIMULATION['a'], SIMULATION['b'], SIMULATION['c'])
    s_sim = s_sim + SIMULATION['sigma'] * np.random.default_rng(123).standard_normal(N)
    ETALON_SIMULE = top_k(s_sim, int(round(SIMULATION['q'] * N)))
    HXBUDDY = {}
    for nom in SIMULATION['sondes']:
        p = SONDES[nom]
        tp = int((p * ETALON_SIMULE).sum())
        tn = int(((1 - p) * (1 - ETALON_SIMULE)).sum())
        f1m = (2 * tp / (p.sum() + ETALON_SIMULE.sum()) + 2 * tn / (2 * N - p.sum() - ETALON_SIMULE.sum())) / 2
        HXBUDDY[nom] = (round(np.mean(p == ETALON_SIMULE), 4), round(f1m, 3))
    print('HxBuddy simulé :', HXBUDDY)

In [ ]:
def normaliser(x):
    return None if x is None else (x / 100 if x > 1 else x)


# HxBuddy affiche « Accuracy 90.83 % » et « F1 macro 90.44 % » : 2 décimales en pourcentage.
ARRONDI_ACC, ARRONDI_F1 = 0.00005, 0.00005
F1_MACRO = True


def q_compatibles(acc, f1, n_pos, macro=F1_MACRO):
    """Valeurs entières de Q (positifs de l'étalon) compatibles avec (accuracy, F1) arrondis."""
    Q = np.arange(1, N)
    ok = np.zeros(len(Q), dtype=bool)
    for juste in range(int(np.ceil((acc - ARRONDI_ACC) * N - 1e-9)), int(np.floor((acc + ARRONDI_ACC) * N + 1e-9)) + 1):
        deux_tp = juste - N + n_pos + Q                  # l'accuracy fixe TP pour chaque Q
        tp = deux_tp / 2
        tn = N - n_pos - Q + tp
        valide = (deux_tp % 2 == 0) & (tp >= 0) & (tn >= 0) & (tp <= np.minimum(n_pos, Q))
        f1_pos = 2 * tp / (n_pos + Q)
        f1_calc = (f1_pos + 2 * tn / (2 * N - n_pos - Q)) / 2 if macro else f1_pos
        ok |= valide & (np.abs(f1_calc - f1) <= ARRONDI_F1 + 1e-9)
    return Q[ok]


OBS = {k: (normaliser(v[0]), normaliser(v[1])) for k, v in HXBUDDY.items()}
if any(n not in HYP.columns for n in OBS):   # sonde ajoutée depuis le dernier calcul
    HYP = table_hypotheses(SONDES)
ENSEMBLES = []
for nom, (acc, f1) in OBS.items():
    if f1 is not None:
        qs = q_compatibles(acc, f1, SONDES[nom].sum())
        ENSEMBLES.append(set(qs))
        print(f'{nom:24s} acc {acc:.4f} F1 {f1:.3f} -> positifs étalon compatibles : '
              + (f'{qs.min()} à {qs.max()} ({qs.min() / N:.1%} à {qs.max() / N:.1%})' if len(qs) else 'aucun'))
Q_OBS = None
if ENSEMBLES:
    communs = sorted(set.intersection(*ENSEMBLES))
    if communs:
        Q_OBS = (min(communs), max(communs))
        print(f"Positifs de l'étalon : entre {Q_OBS[0]} et {Q_OBS[1]} ({Q_OBS[0] / N:.1%} à {Q_OBS[1] / N:.1%})")
    else:
        print('Aucun Q commun à toutes les sondes : vérifiez la saisie (accuracy/F1 inversés ? mauvaise sonde ?)')
if not OBS:
    print('Aucun retour HxBuddy saisi : seule l\'ancre 0,270 contraint les hypothèses. Ne soumettez pas encore le choix final.')

In [ ]:
TOL_ACC, TOL_ECART = 0.002, 0.012


def ajuster(hyp, obs, poids_ancre=1.0, q_obs=None):
    perte = poids_ancre * ((hyp.ecart_base - ECART_BASE) / TOL_ECART) ** 2
    for nom, (acc, _) in obs.items():
        perte = perte + ((hyp[nom] - acc) / TOL_ACC) ** 2
    if q_obs:   # hors de l'intervalle décodé : pénalité quadratique (0,5 % de taux = 1 unité)
        k = hyp.q * N
        hors = np.maximum(q_obs[0] - k, 0) + np.maximum(k - q_obs[1], 0)
        perte = perte + (hors / (0.005 * N)) ** 2
    w = np.exp(-(perte - perte.min()) / 2)
    return hyp.assign(perte=perte, poids=w / w.sum()).sort_values('perte')


POST = ajuster(HYP, OBS, q_obs=Q_OBS)
PLAUSIBLES = POST[POST.perte <= POST.perte.min() + 6]
cols = ['a', 'b', 'c', 'sigma', 'q', 'ecart_base', 'perte', *OBS]
print(f'{len(PLAUSIBLES):,} hypothèses plausibles sur {len(POST):,}. Les meilleures :')
print(POST[cols].head(12).round(3).to_string(index=False))
print(f'\nMasse a posteriori : sigma=0 {POST.loc[POST.sigma == 0, "poids"].sum():.0%} | '
      f'q moyen {(POST.q * POST.poids).sum():.3f} | a moyen {(POST.a * POST.poids).sum():.3f} | '
      f'b moyen {(POST.b * POST.poids).sum():.2f} | c moyen {(POST.c * POST.poids).sum():.2f}')

In [ ]:
# Probabilité, pour chaque candidat, d'être positif dans l'étalon (moyenne sur les hypothèses plausibles).
if len(PLAUSIBLES) > 300:   # tirage a posteriori avec remise ; poids = fréquence de tirage
    freq = PLAUSIBLES.sample(400, weights='poids', replace=True, random_state=0).index.value_counts()
    TOP = PLAUSIBLES.loc[freq.index].assign(poids=(freq / freq.sum()).to_numpy())
else:
    TOP = PLAUSIBLES.assign(poids=PLAUSIBLES.poids / PLAUSIBLES.poids.sum())


def proba_etalon(row):
    s0 = score(row.a, row.b, row.c)
    k = int(round(row.q * N))
    tirages = [s0] if row.sigma == 0 else [s0 + row.sigma * e for e in BRUIT]
    return np.mean([top_k(s, k) for s in tirages], axis=0)


REFS = np.array([proba_etalon(row) for row in TOP.itertuples()])   # [hypothèse, candidat]
PW = TOP.poids.to_numpy()
p_ref = PW @ REFS


def attendu(pred, par_hyp=False):
    """Accuracy, écart EO et points techniques (/35) espérés sous les hypothèses retenues."""
    acc = (REFS @ pred + (1 - REFS) @ (1 - pred)) / N
    tp_c = REFS @ (pred * (g == 0)) / (REFS @ (g == 0))
    tp_r = REFS @ (pred * (g == 1)) / (REFS @ (g == 1))
    ecart = tp_c - tp_r
    q, t = REFS.mean(1), pred.mean()
    acc_hasard = q * t + (1 - q) * (1 - t)
    util = np.clip((acc - acc_hasard) / (1 - acc_hasard), 0, 1)
    pts = 15 * util + 20 * np.clip(1 - np.abs(ecart) / ECART_BASE, 0, 1)
    ok = BUDGET_MIN <= t <= BUDGET_MAX
    if par_hyp:
        return acc, ecart, pts * ok
    return PW @ acc, PW @ ecart, (PW @ pts) * ok


# Décision : top-k par probabilité ; k et le partage centres/éloignées choisis pour maximiser les points espérés.
ordre = np.argsort(-p_ref, kind='stable')
meilleur = None
for k in range(K_MIN, K_MAX + 1, 10):
    pred = np.zeros(N, dtype=int); pred[ordre[:k]] = 1
    for decal in range(-30, 31, 5):   # >0 : on déplace des octrois des centres vers les éloignées
        p2 = pred.copy()
        if decal:
            src, dst = (g == 0, g == 1) if decal > 0 else (g == 1, g == 0)
            out = [i for i in ordre[:k][::-1] if src[i]][:abs(decal)]
            inn = [i for i in ordre[k:] if dst[i]][:abs(decal)]
            p2[out] = 0; p2[inn] = 1
        res = attendu(p2)
        if meilleur is None or res[2] > meilleur[0][2]:
            meilleur = (res, k, decal, p2)
(acc_f, ecart_f, pts_f), K_FINAL, DECAL, FINAL = meilleur
print(f'Choix : k={K_FINAL} ({K_FINAL / N:.1%}), décalage {DECAL:+d} | accuracy espérée {acc_f:.3f} | '
      f'écart EO espéré {ecart_f:+.3f} | points techniques espérés {pts_f:.1f}/35')
for nom in OBS:
    a_, e_, p_ = attendu(SONDES[nom])
    print(f'  pour comparaison {nom:22s} accuracy espérée {a_:.3f} | écart {e_:+.3f} | points {p_:.1f}/35')

# Garde-fou : on ne remplace la meilleure sonde déjà mesurée que si le gain en points (/35) est sûr,
# c'est-à-dire positif sous au moins 80 % de la masse des hypothèses plausibles (comparaison appariée).
dans_budget = [n for n in OBS if BUDGET_MIN <= SONDES[n].mean() <= BUDGET_MAX]
RECOMMANDE = 'FINAL'
if dans_budget:
    meilleure = max(dans_budget, key=lambda n: attendu(SONDES[n])[2])
    diff = attendu(FINAL, par_hyp=True)[2] - attendu(SONDES[meilleure], par_hyp=True)[2]
    sur = PW @ (diff > 0)
    print(f'Gain attendu sur {meilleure} : {PW @ diff:+.2f} points /35, positif dans {sur:.0%} des hypothèses')
    if sur < 0.8:
        RECOMMANDE = meilleure
        FINAL = SONDES[meilleure]
        print(f"Gain pas encore sûr : predictions.csv = {meilleure}. "
              f"Soumettez la sonde suivante pour réduire l'incertitude.")

In [ ]:
# Sonde suivante : celle dont l'accuracy prédite varie le plus entre les hypothèses encore plausibles.
cands = {}
for a in A_GRID[::2]:
    for b in B_GRID[::2]:
        for c in C_GRID:
            for k in (1600, 1750):
                cands[f'a{a}_b{b}_c{c}_k{k}'] = top_k(score(a, b, c), k)
C = np.array(list(cands.values()))
acc_pred = (C @ REFS.T + (1 - C) @ (1 - REFS).T) / N           # [candidat, hypothèse]
moy = acc_pred @ PW
std = np.sqrt(((acc_pred - moy[:, None]) ** 2) @ PW)
i = int(np.argmax(std))
SONDE_SUIVANTE = list(cands)[i]
pd.DataFrame({'id_candidat': candidats.id_candidat, 'decision_octroi': C[i]}) \
    .to_csv(f'sondes/suivante_{SONDE_SUIVANTE}.csv', index=False)
SONDES[f'suivante_{SONDE_SUIVANTE}'] = C[i]
print(f'Sonde suivante : sondes/suivante_{SONDE_SUIVANTE}.csv | écart-type prédit {std[i]:.3f} '
      f'(rien à apprendre si < 0.003)')

In [ ]:
pd.DataFrame({'id_candidat': candidats.id_candidat, 'decision_octroi': FINAL}).to_csv('predictions.csv', index=False)
verif = pd.read_csv('predictions.csv')
assert len(verif) == N and set(verif.decision_octroi.unique()) <= {0, 1}
assert verif.id_candidat.tolist() == candidats.id_candidat.tolist()
taux = verif.decision_octroi.mean()
assert BUDGET_MIN <= taux <= BUDGET_MAX, taux
print(f'predictions.csv OK : {N} lignes, octroi {taux:.1%} | centres {FINAL[g == 0].mean():.1%} | '
      f'éloignées {FINAL[g == 1].mean():.1%}')

if ETALON_SIMULE is not None:
    def vrai(pred):
        e = pred[(ETALON_SIMULE == 1) & (g == 0)].mean() - pred[(ETALON_SIMULE == 1) & (g == 1)].mean()
        return np.mean(pred == ETALON_SIMULE), e
    print('Vérité simulée :')
    for nom, pred in [('FINAL', FINAL), *[(n, SONDES[n]) for n in OBS]]:
        acc_v, e_v = vrai(pred)
        print(f'  {nom:24s} accuracy réelle {acc_v:.3f} | écart EO réel {e_v:+.3f}')

In [ ]:
# Front de Pareto : on retire la pénalité régionale par fraction alpha (0 = comité, 1 = neutralisé,
# > 1 = surcorrection). Écart signé : au-delà de zéro, les régions éloignées sont favorisées.
import matplotlib.pyplot as plt

alphas = np.round(np.linspace(0, 1.5, 16), 2)
front = []
for al in alphas:
    pred = top_k(score(A_COMITE, B_COMITE) + (1 - al) * PENALITE * g, 1600)
    acc_, ecart_, pts_ = attendu(pred)
    front.append((al, ecart_, acc_, pts_))
front = pd.DataFrame(front, columns=['alpha', 'ecart_EO', 'accuracy', 'points'])
print(front.round(3).to_string(index=False))

BLEU, ORANGE, GRIS, ENCRE, ENCRE2 = '#2a78d6', '#eb6834', '#8a8984', '#0b0b0b', '#52514e'
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(front.ecart_EO, front.accuracy, color=BLEU, lw=2, marker='o', ms=6,
        markeredgecolor='white', markeredgewidth=1.5, label='Retrait progressif de la pénalité (α)')
for _, row in front.iloc[[0, 5, 10, 15]].iterrows():
    ax.annotate(f'α={row.alpha:g}', (row.ecart_EO, row.accuracy), textcoords='offset points',
                xytext=(0, 9), ha='center', fontsize=9, color=ENCRE2)
ax.axvline(0, color='#c9c8c2', lw=1, zorder=0)
b_acc, b_ecart, _ = attendu(base)
ax.scatter([b_ecart], [b_acc], s=60, color=GRIS, edgecolor='white', linewidth=1.5, zorder=3,
           label='Modèle de base')
ax.scatter([ecart_f], [acc_f], s=90, marker='D', color=ORANGE, edgecolor='white', linewidth=1.5,
           zorder=4, label='Soumission retenue')
ax.set_xlabel("Écart d'égalité des chances : TPR centres − TPR éloignées (espéré, 0 = équité)", color=ENCRE2)
ax.set_ylabel("Accord avec l'étalon (espéré)", color=ENCRE2)
ax.set_title("Front de Pareto équité / utilité", loc='left', color=ENCRE, fontsize=12)
ax.grid(color='#e6e5e0', lw=0.8); ax.set_axisbelow(True)
for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)
for sp in ['left', 'bottom']:
    ax.spines[sp].set_color('#c9c8c2')
ax.tick_params(colors=ENCRE2)
ax.legend(frameon=False, fontsize=9, loc='lower left')
plt.tight_layout()
plt.savefig('pareto.png', dpi=150)
plt.show()
print(f'Temps total : {time.time() - T0:.1f} s')